### Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import plotly.express as px
import requests
import time
import xml.etree.ElementTree as ET
from sklearn.preprocessing import MinMaxScaler
### multiple testing
from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests


import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap

### Data import

#### DrugBank

In [ ]:
# import xml.etree.ElementTree as ET

# # Load XML
# drugbank_xml = 'Data/DGIDB/drug_bank.xml'
# tree = ET.parse(drugbank_xml)
# root = tree.getroot()

# # Namespace
# ns = {'db': 'http://www.drugbank.ca'}

# Helper to clean tag names
def clean_tag(tag):
    return tag.split('}')[-1] if '}' in tag else tag

# Recursive function to print structure
def print_structure(elem, level=0):
    indent = '  ' * level
    print(f"{indent}- {clean_tag(elem.tag)}")
    for child in elem:
        print_structure(child, level + 1)

# # Get first drug
# first_drug = root.find('db:drug', ns)

# print("🌿 Structure of First Drug Entry:")
# print_structure(first_drug)
# print("\n🌳 Structure of First 3 Drug Entries:")
# drugs = root.findall('db:drug', ns)

# for i, drug in enumerate(drugs[:3]):
#     print(f"\n🔬 Drug {i+1}:")
#     print_structure(drug)


In [ ]:
def structure_drug_bank_data(drug_bank_file = 'Data/DGIDB/drug_bank.xml'):
    """
    Function to structure the drug bank data from the XML file.
    :param drug_bank_file: Path to the drug bank XML file.
    :return: DataFrame containing structured drug bank data.
    """
    ### FYI the .find command only finds the first instance of a tag, 
    ### while .findall retrieves all instances of the specified tag within the current element.

    tree = ET.parse(drug_bank_file)
    root = tree.getroot()

    # DrugBank uses a specific namespace
    ns = {'db': 'http://www.drugbank.ca'}
    ### extract all drug elements
    drugs = root.findall('db:drug', ns)
    print(f"Found {len(drugs)} drugs in the DrugBank XML.")
    # Extract drug-gene interactions
    interactions = []
    # The interactions list will store dictionaries with 'drug' and 'gene' keys.
    for drug in root.findall('db:drug', ns): # root.findall('db:drug', ns): Finds all <drug> elements using the namespace.
        drug_name  = drug.find('db:name', ns).text  # drug.find('db:name', ns): Gets the drug's name.
        # print(drug_name)
        for target in drug.findall('db:targets/db:target', ns):  # drug.findall('db:targets/db:target', ns): Finds all <target> elements within <targets>.
            # print(target.tag)
            gene_description = target.find('db:name', ns)  # target.find('db:name', ns): Extracts the gene name for each target.
            poly = target.find('db:polypeptide', ns)  # target.find('db:polypeptide', ns): Extracts the polypeptide information.
            action = target.find('db:actions/db:action', ns) # target.find('db:actions/db:action', ns): Extracts the action of the drug on the target.
            if poly is not None:
                poly_name = poly.find('db:name', ns)
                gene_name = poly.find('db:gene-name', ns)
                specific_function = poly.find('db:specific-function', ns)
                interactions.append({
                    'drug': drug_name,
                    'polypeptide': poly_name.text if poly_name is not None else None,
                    'gene': gene_name.text if gene_name is not None else None,
                    'gene_description': gene_description.text if gene_description is not None else None,
                    'action': action.text if action is not None else None,
                    'specific_function': specific_function.text if specific_function is not None else None
                })
            ############# if polypeptide is not present, we still want to add the drug and gene information
            ############# this is because some drugs may not have a polypeptide associated with them
            ############# but we still want to capture the drug and gene information
            ############# this is common in the DrugBank database, where some drugs target genes directly
            ############# and do not have a polypeptide associated with them

            else:
                gene_name = None
                specific_function = None
                poly_name = None
                action = None
                gene_description = None
                resource = None
                identifier = None
  
                interactions.append({
                        'drug': drug_name,
                        'polypeptide': poly_name.text if poly_name is not None else None,
                        'gene': gene_name.text if gene_name is not None else None,
                        'gene_description': gene_description.text if gene_description is not None else None,
                        'action': action.text if action is not None else None,
                        'specific_function': specific_function.text if specific_function is not None else None
                    })
        
    # Convert to DataFrame
    # Converts the list of dictionaries into a pandas DataFrame, which is easier to analyze, filter, and export.
    df = pd.DataFrame(interactions)

    return df

In [ ]:
Drug_bank = structure_drug_bank_data('Data/DGIDB/drug_bank.xml')

In [ ]:
Drug_bank

In [ ]:
### Drug bank statistics
print(f"Number of unique medications in drugbank {Drug_bank['drug'].nunique()}")
print(f"Number of unique genes available in drugBank {Drug_bank['gene'].nunique()}")
print(f"Number of interactions within drugbank {len(Drug_bank)}")

#### PPI

In [ ]:
protein_interaction = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.v12.0.txt', sep= ' ')
protein_interaction_full = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.full.v12.0.txt', sep= ' ')
protein_interaction_detailed = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.detailed.v12.0.txt', sep= ' ')
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

In [ ]:
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

# Method 1: Using the to_dict() method with 'index' as orient
protein_info_translate_name_dict = protein_info.set_index('#string_protein_id')['preferred_name'].to_dict()
protein_alias_translate_name_dict = protein_aliases.set_index('#string_protein_id')['alias'].to_dict()
#print(protein_info_translate_name_dict)

### Protein1
protein1_name = []
for prot_id in tqdm(protein_interaction['protein1']):
    if prot_id in protein_info_translate_name_dict:
        protein1_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein1_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein1_name.append('')

### Protein 2
protein2_name = []
for prot_id in tqdm(protein_interaction['protein2']):
    if prot_id in protein_info_translate_name_dict:
        protein2_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein2_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein2_name.append('')

protein_interaction['Translated_protein_1'] = protein1_name
protein_interaction['Translated_protein_2'] = protein2_name

# Create a set of all (protein1, protein2) pairs
ppi_pairs = set(zip(protein_interaction['Translated_protein_1'], protein_interaction['Translated_protein_2']))
# Check for missing reverse pairs
missing_reverse = []
for a, b in ppi_pairs:
    if (b, a) not in ppi_pairs:
        missing_reverse.append((a, b))

print(f"Number of pairs missing their reverse: {len(missing_reverse)}")
if missing_reverse:
    print("Examples:", missing_reverse[:10])
else:
    print("All pairs have their reverse present.")

In [ ]:
protein_interaction

#### SOM

In [ ]:
### Somatic mutation data
somatic_mutation = pd.read_csv('Data/TCGA/SOM/cohortMAF.2025-11-12.maf', sep= '\t')
somatic_mutation['Case ID'] = somatic_mutation['Tumor_Sample_Barcode'].str.split('-').str[0:3].str.join('-').tolist()

In [ ]:
### Pull procedures through ICD translation
def query_UMLS(query, codingSystem):
    query = str.upper(query)
    codingSystem = str.upper(codingSystem)
    base_url = 'https://uts-ws.nlm.nih.gov/rest'
    apikey = '6ff64660-bd02-43d6-8b7e-506cbba7242e'
    endpoint = f'content/current/source/{codingSystem}/{query}?'
    apiQuery = f'{base_url}/{endpoint}apiKey={apikey}'
    #print(apiQuery)
    resp = requests.get(apiQuery)
    return resp.json()['result']['name']                                                                                                                         

codingSystems = ['CPT', 'HCPT', 'HCPCS', 'OT', 'ICD9CM', 'ICD10CM']

def translateICD(query):
    codingSystems = ['CPT', 'HCPT', 'HCPCS', 'OT', 'ICD9CM', 'ICD10CM']
    descript = np.nan
    for code in codingSystems:
        try:
            descript = query_UMLS(str(query), code)
            break
        except:
            continue
    return descript

### Data viewing

#### Protein protein interaction network

In [ ]:
protein_interaction

In [ ]:
print(len(set(protein_interaction['protein1'])))
print(len(set(protein_interaction['protein2'])))

### Somatic mutation data

In [ ]:
somatic_mutation

In [ ]:
### identify number of unique patients in the cohort
print(len(set(somatic_mutation['Tumor_Sample_Barcode'])))
### identify number of unique genes in the cohort
print(len(set(somatic_mutation['Hugo_Symbol'])))
### identify number of unique cases in the cohort
print(len(set(somatic_mutation['Case ID'])))


### CNV data

In [ ]:
sample_sheet = pd.read_csv('Data/TCGA/Gene level CNV/gdc_sample_sheet.2025-11-03.tsv', sep= '\t')
manifest = pd.read_csv('Data/TCGA/Gene level CNV/gdc_manifest.2025-11-03.174310.txt', sep= '\t')

In [ ]:
### get length of Gene level CNV folder
cnv_folder = 'Data/TCGA/Gene level CNV/'
cnv_files = os.listdir(cnv_folder)
print(len(cnv_files))

In [ ]:
manifest

In [ ]:
sample_sheet

In [ ]:
len(set(sample_sheet['File Name']))

In [ ]:
len(set(sample_sheet['File ID']))

In [ ]:
### split case id values by comma and and to list
case_ids = []
ids = sample_sheet['Case ID'].unique()
for cases in ids:
    if ',' in cases:
        cases = cases.split(',')
        for case2 in cases:
            case_ids.append(case2.strip())
    else:
        case_ids.append(cases)
    # print(case_ids)
print(len(set(case_ids)))

In [ ]:
case_ids =len(set(case_ids))
print(case_ids)

In [ ]:
manifest

### Clinical data

In [ ]:
clinical_data = pd.read_csv('Data/TCGA/Gene level CNV/clinical.tsv', sep= '\t')

In [ ]:
for col in clinical_data.columns:
    print(col)

In [ ]:
icd_10 = clinical_data['diagnoses.icd_10_code']
icd_10_value_counts = icd_10.value_counts().reset_index()
icd_10_value_counts.columns = ['ICD-10 Code', 'Count']
###drop the '--' code
icd_10_value_counts = icd_10_value_counts[icd_10_value_counts['ICD-10 Code'] != "'--"]
icd_10_value_counts = icd_10_value_counts[icd_10_value_counts['ICD-10 Code'] != '']
### plot icd_10 code
plt.figure(figsize=(10, 6))
plt.bar(icd_10_value_counts['ICD-10 Code'], icd_10_value_counts['Count'], color='skyblue')
plt.xlabel('ICD-10 Code')
plt.ylabel('Count')
plt.title('Distribution of ICD-10 Codes')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
### create translation dictionary for ICD-10 codes to description
icd_10_translation = {}
for code in tqdm(icd_10_value_counts['ICD-10 Code']):
    ### double check if code in dictionary already
    if code not in icd_10_translation:
        description = translateICD(code)
        icd_10_translation[code] = description

In [ ]:
### redo graph with translated ICD codes
icd_10_value_counts['Description'] = icd_10_value_counts['ICD-10 Code'].map(icd_10_translation)
### Truncate long descriptions
max_length = 40
icd_10_value_counts['Description_Short'] = icd_10_value_counts['Description'].apply(
    lambda x: x[:max_length] + '...' if isinstance(x, str) and len(x) > max_length else x
)
plt.figure(figsize=(12, 8))
plt.bar(icd_10_value_counts['Description_Short'], icd_10_value_counts['Count'], color='skyblue')
plt.xlabel('ICD-10 Description')
plt.ylabel('Patient Count')
plt.title('Distribution of ICD-10')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
### table of ICD-10 codes with descriptions, and counts
icd_10_value_counts[['ICD-10 Code', 'Description', 'Count']]

In [ ]:
clinical_data

In [ ]:
### age distribution
ages = clinical_data['demographic.age_at_index']
ages = ages.dropna()
ages = ages[ages != "'--"]
ages = ages.astype(int)
plt.figure(figsize=(10, 6))
plt.hist(ages, bins=30, color='blue', edgecolor='black')
plt.xlabel('Age at Diagnosis')
plt.ylabel('Frequency')
plt.title('Age Distribution at Diagnosis')
plt.grid(axis='y', alpha=0.75)
plt.tight_layout()
# ### add vertical bars for wehre the majority of ages lie, line for showing q1 and q3
plt.axvline(ages.median(), color='red', linestyle='--', linewidth=2, label=f'Median Age: {ages.median()}')
plt.axvline(ages.quantile(0.25), color='green', linestyle='--', linewidth=2, label=f'Q1 Age: {ages.quantile(0.25)}')
plt.axvline(ages.quantile(0.75), color='orange', linestyle='--', linewidth=2, label=f'Q3 Age: {ages.quantile(0.75)}')
plt.legend()
plt.show()


In [ ]:
len(clinical_data)

In [ ]:
clinical_data = pd.read_csv('Data/TCGA/Gene level CNV/clinical.tsv', sep= '\t')

In [ ]:
for col in clinical_data.columns:
    print(col)

In [ ]:
### plot gender distribution in pie chart
genders = clinical_data['demographic.gender']
gender_counts = pd.Series(genders).value_counts()
plt.figure(figsize=(8, 8))
plt.pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%', startangle=90)
### increase font size of title
plt.title('Gender Distribution of Patients', fontsize=16)
### increase font size of data labels
plt.setp(plt.gca().texts, fontsize=16)
plt.axis('equal')
plt.show()